[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ContextLab/experimental-psychology/blob/main/notebooks/binomial_demo.ipynb)

# Binomial Distribution Demo

This notebook demonstrates how the binomial distribution works and how we can use it to calculate p-values. These concepts are foundational for the statistical tests you'll use in your labs.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

## What is the binomial distribution?

The **binomial distribution** models the number of successes in a fixed number of independent trials, where each trial has the same probability of success.

It's defined by two parameters:
- **n**: the number of trials
- **p**: the probability of success on each trial

For example, if you flip a fair coin 30 times, the number of heads you get follows a binomial distribution with n=30 and p=0.5.

In [ ]:
# Parameters for our binomial distribution
n = 30   # number of trials (e.g., 30 coin flips)
p = 0.5  # probability of success on each trial (fair coin)

# All possible numbers of successes (0 through 30)
k = np.arange(0, n + 1)

# Calculate the probability of each outcome using the PMF
# (probability mass function)
pmf = stats.binom.pmf(k, n, p)

# Plot the distribution
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(k, pmf, color='steelblue', edgecolor='white')
ax.set_xlabel('Number of Successes (k)', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title(f'Binomial Distribution (n={n}, p={p})', fontsize=14)
ax.set_xticks(range(0, n + 1, 5))
plt.tight_layout()
plt.show()

## What is a p-value?

A **p-value** is the probability of observing a result **as extreme as (or more extreme than)** what we actually got, *assuming the null hypothesis is true*.

For our coin-flipping example:
- **Null hypothesis**: the coin is fair (p = 0.5)
- **Observation**: we got some number of heads
- **p-value**: how likely is it to get a result this far (or farther) from the expected value of 15, if the coin really is fair?

A small p-value means the result would be surprising under the null hypothesis, giving us evidence against it.

In [ ]:
# Suppose we observed 20 successes out of 30 trials
observed = 20
expected = n * p  # expected number of successes under H0

# For a two-tailed test, we need the probability of results
# as extreme as ours in EITHER direction.
# Our result is 20, which is 5 above the expected value of 15.
# So we also count values 5 or more BELOW the expected value (i.e., <= 10).
deviation = abs(observed - expected)
lower_tail = expected - deviation  # 10
upper_tail = expected + deviation  # 20

# Two-tailed p-value: P(X <= lower_tail) + P(X >= upper_tail)
p_value = stats.binom.cdf(lower_tail, n, p) + (1 - stats.binom.cdf(upper_tail - 1, n, p))

print(f'Observed: {observed} successes out of {n} trials')
print(f'Expected under H0: {expected}')
print(f'Two-tailed p-value: {p_value:.4f}')

# Visualize: shade the tails of the distribution
fig, ax = plt.subplots(figsize=(10, 5))

# Color each bar: red if in a tail, blue otherwise
colors = ['firebrick' if (ki <= lower_tail or ki >= upper_tail) else 'steelblue'
          for ki in k]
ax.bar(k, pmf, color=colors, edgecolor='white')

# Mark the observed value
ax.axvline(observed, color='black', linestyle='--', linewidth=1.5,
           label=f'Observed = {observed}')

ax.set_xlabel('Number of Successes (k)', fontsize=12)
ax.set_ylabel('Probability', fontsize=12)
ax.set_title(f'Two-Tailed p-value = {p_value:.4f}\n'
             f'(shaded regions are as extreme or more extreme than our result)',
             fontsize=13)
ax.legend(fontsize=11)
ax.set_xticks(range(0, n + 1, 5))
plt.tight_layout()
plt.show()

## Connection to the survey lab

In your survey lab, you'll be testing whether two variables are related -- for example, whether people who prefer cats over dogs also tend to prefer tea over coffee.

Under the **null hypothesis** (no relationship between the variables), any apparent pattern in the data is just due to chance. The statistical test you run is essentially asking:

> *"If there were really no relationship, how surprising is the pattern we observed?"*

If the p-value is small (typically less than 0.05), we conclude that the result is unlikely to have occurred by chance alone, and we have evidence for a real relationship.

The binomial distribution is the simplest version of this logic. More complex tests (like chi-squared) work on the same principle but can handle more than two categories.

In [ ]:
# Bonus: What does the distribution of p-values look like
# when the null hypothesis IS true?

# Simulate 1000 experiments, each with 30 fair coin flips
np.random.seed(42)
n_experiments = 1000
n_trials = 30
p_null = 0.5

# Run all experiments at once
results = np.random.binomial(n_trials, p_null, size=n_experiments)

# Calculate the two-tailed p-value for each experiment
p_values = []
for obs in results:
    # Two-tailed p-value using the binomial test
    test_result = stats.binomtest(obs, n_trials, p_null, alternative='two-sided')
    p_values.append(test_result.pvalue)

p_values = np.array(p_values)

# Plot the distribution of p-values
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(p_values, bins=20, color='steelblue', edgecolor='white', density=True)
ax.axhline(1.0, color='firebrick', linestyle='--', linewidth=1.5,
           label='Uniform (expected under H0)')
ax.axvline(0.05, color='orange', linestyle='-', linewidth=2,
           label='p = 0.05 threshold')
ax.set_xlabel('p-value', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Distribution of p-values under the null hypothesis (1000 simulations)',
             fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

# How many experiments had p < 0.05?
n_significant = np.sum(p_values < 0.05)
pct_significant = 100 * n_significant / n_experiments
print(f'{n_significant} out of {n_experiments} experiments ({pct_significant:.1f}%) '
      f'had p < 0.05')
print(f'This is close to 5%, which is exactly what we\'d expect by chance!')